In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Pasy Transpilatora AI
Pasy Transpilatora zasilane przez AI to pasy, które działają jako zamiennik tradycyjnych pasów Qiskit dla niektórych zadań transpilacji. Często dają lepsze wyniki niż istniejące algorytmy heurystyczne (takie jak mniejsza głębokość obwodu i liczba bramek CNOT), a jednocześnie są znacznie szybsze niż algorytmy optymalizacji, takie jak rozwiązywacze spełniania formuł boolowskich. Pasy Transpilatora AI mogą działać w środowisku lokalnym lub w chmurze za pomocą Qiskit Transpiler Service, jeśli korzystasz z planu IBM Quantum&reg; Premium Plan, Flex Plan lub On-Prem (za pośrednictwem IBM Quantum Platform API) Plan.

> **Note:** Pasy Transpilatora zasilane przez AI są w wersji beta i mogą ulec zmianom.
>     Jeśli masz uwagi lub chcesz skontaktować się z zespołem deweloperskim, skorzystaj z tego [kanału Qiskit Slack Workspace](https://qiskit.slack.com/archives/C06KF8YHUAU).

Aktualnie dostępne są następujące pasy:

**Pasy trasowania**
 - `AIRouting`: Wybór układu i trasowanie Circuit

**Pasy syntezy Circuit**
 - `AICliffordSynthesis`: Synteza Circuit Clifforda
 - `AILinearFunctionSynthesis`: Synteza Circuit funkcji liniowych
 - `AIPermutationSynthesis`: Synteza Circuit permutacji
 - `AIPauliNetworkSynthesis`: Synteza Circuit sieci Pauliego

Aby korzystać z pasów Transpilatora AI, najpierw [zainstaluj pakiet `qiskit-ibm-transpiler`](/guides/qiskit-transpiler-service#install-transpiler-service). Odwiedź [dokumentację API qiskit-ibm-transpiler](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler), aby uzyskać więcej informacji o dostępnych opcjach.

## Uruchamianie pasów Transpilatora AI lokalnie lub w chmurze
Jeśli chcesz bezpłatnie używać pasów Transpilatora zasilanych przez AI w swoim lokalnym środowisku, zainstaluj `qiskit-ibm-transpiler` z dodatkowymi zależnościami w następujący sposób:

In [1]:
from qiskit.transpiler import PassManager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_runtime import QiskitRuntimeService

backend = QiskitRuntimeService().backend("ibm_torino")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=backend,
            optimization_level=2,
            layout_mode="optimize",
            local_mode=True,
        )
    ]
)


circuit = efficient_su2(101, entanglement="circular", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

Bez tych dodatkowych zależności pasy Transpilatora zasilane przez AI działają w chmurze za pośrednictwem Qiskit Transpiler Service (dostępne tylko dla użytkowników planu IBM Quantum Premium Plan, Flex Plan lub On-Prem (za pośrednictwem IBM Quantum Platform API) Plan). Po zainstalowaniu dodatkowych zależności domyślnym trybem uruchamiania pasów Transpilatora zasilanych przez AI jest korzystanie z lokalnego komputera.

## Pas trasowania AI
Pas `AIRouting` pełni zarówno rolę etapu układu, jak i etapu trasowania. Można go użyć w ramach `PassManager` w następujący sposób:

In [ ]:
from qiskit.transpiler import PassManager

from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_transpiler.ai.synthesis import AILinearFunctionSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectLinearFunctions
from qiskit_ibm_transpiler.ai.synthesis import AIPauliNetworkSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectPauliNetworks
from qiskit.circuit.library import efficient_su2

ibm_torino = QiskitRuntimeService().backend("ibm_torino")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=ibm_torino,
            optimization_level=3,
            layout_mode="optimize",
            local_mode=True,
        ),  # Route circuit
        CollectLinearFunctions(),  # Collect Linear Function blocks
        AILinearFunctionSynthesis(
            backend=ibm_torino, local_mode=True
        ),  # Re-synthesize Linear Function blocks
        CollectPauliNetworks(),  # Collect Pauli Networks blocks
        AIPauliNetworkSynthesis(
            backend=ibm_torino, local_mode=True
        ),  # Re-synthesize Pauli Network blocks.
    ]
)

circuit = efficient_su2(10, entanglement="full", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

Tutaj `backend` określa, dla której mapy sprzężeń przeprowadzać trasowanie, `optimization_level` (1, 2 lub 3) określa nakład obliczeniowy przeznaczony na ten proces (wyższy poziom zazwyczaj daje lepsze wyniki, ale trwa dłużej), a `layout_mode` określa sposób obsługi wyboru układu.
`layout_mode` oferuje następujące opcje:

- `keep`: Respektuje układ ustawiony przez poprzednie pasy Transpilatora (lub używa trywialnego układu, jeśli nie jest ustawiony). Zwykle używany tylko wtedy, gdy Circuit musi być uruchamiany na konkretnych Qubitach urządzenia. Często daje gorsze wyniki, ponieważ ma mniej miejsca na optymalizację.
- `improve`: Używa układu ustawionego przez poprzednie pasy Transpilatora jako punktu startowego. Jest przydatny, gdy masz dobre wstępne przypuszczenie co do układu; na przykład dla Circuit zbudowanych w sposób, który w przybliżeniu odwzorowuje mapę sprzężeń urządzenia. Jest też przydatny, gdy chcesz wypróbować inne specyficzne pasy układu w połączeniu z pasem `AIRouting`.
- `optimize`: Jest to tryb domyślny. Działa najlepiej dla ogólnych Circuit, gdy nie masz dobrych przypuszczeń co do układu. Ten tryb ignoruje poprzednie wybory układu.
- `local_mode`: Ta flaga określa, gdzie uruchamiany jest pas `AIRouting`. Jeśli `False`, `AIRouting` działa zdalnie za pośrednictwem Qiskit Transpiler Service. Jeśli `True`, pakiet próbuje uruchomić pas w lokalnym środowisku, z możliwością awaryjnego przełączenia na tryb chmury, jeśli wymagane zależności nie zostaną znalezione.

## Pasy syntezy Circuit AI
Pasy syntezy Circuit AI umożliwiają optymalizację fragmentów różnych typów Circuit ([Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford), [Linear Function](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction), [Permutation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation), Pauli Network) poprzez ich resyntetyzowanie. Typowy sposób użycia pasu syntezy jest następujący:

In [ ]:
from qiskit_ibm_transpiler import generate_ai_pass_manager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import QiskitRuntimeService


backend = QiskitRuntimeService().backend("ibm_torino")
torino_coupling_map = backend.coupling_map


su2_circuit = efficient_su2(101, entanglement="circular", reps=1)

ai_transpiler_pass_manager = generate_ai_pass_manager(
    coupling_map=torino_coupling_map,
    ai_optimization_level=3,
    optimization_level=3,
    ai_layout_mode="optimize",
)

ai_su2_transpiled_circuit = ai_transpiler_pass_manager.run(su2_circuit)

Synteza respektuje mapę sprzężeń urządzenia: można ją bezpiecznie uruchomić po innych pasach trasowania bez zakłócania Circuit, dzięki czemu ogólny Circuit nadal będzie zgodny z ograniczeniami urządzenia. Domyślnie synteza zastąpi oryginalny pod-Circuit tylko wtedy, gdy zsyntetyzowany pod-Circuit jest lepszy od oryginału (obecnie sprawdzana jest tylko liczba bramek CNOT), ale można wymusić zawsze zastępowanie Circuit ustawiając `replace_only_if_better=False`.

Następujące pasy syntezy są dostępne w `qiskit_ibm_transpiler.ai.synthesis`:

- *AICliffordSynthesis*: Synteza dla Circuit [Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford) (bloki bramek `H`, `S` i `CX`). Aktualnie obsługuje bloki do dziewięciu Qubitów.
- *AILinearFunctionSynthesis*: Synteza dla Circuit [Linear Function](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction) (bloki bramek `CX` i `SWAP`). Aktualnie obsługuje bloki do dziewięciu Qubitów.
- *AIPermutationSynthesis*: Synteza dla Circuit [Permutation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation) (bloki bramek `SWAP`). Aktualnie dostępna dla bloków 65, 33 i 27 Qubitów.
- *AIPauliNetworkSynthesis*: Synteza dla Circuit sieci Pauliego (bloki bramek `H`, `S`, `SX`, `CX`, `RX`, `RY` i `RZ`). Aktualnie obsługuje bloki do sześciu Qubitów.

Oczekujemy stopniowego zwiększania rozmiaru obsługiwanych bloków.

Wszystkie pasy używają puli wątków do równoległego wysyłania kilku żądań. Domyślnie maksymalna liczba wątków to liczba rdzeni plus cztery (wartości domyślne dla obiektu Python `ThreadPoolExecutor`). Możesz jednak ustawić własną wartość za pomocą argumentu `max_threads` podczas tworzenia instancji pasu. Na przykład poniższa linia tworzy instancję pasu `AILinearFunctionSynthesis`, pozwalając mu używać maksymalnie 20 wątków.